In [1]:
from pathlib import Path
print(Path.cwd())

/Users/seiyam/Documents/GitHub/data_viz/static/src


In [2]:
import pandas as pd
import numpy as np
import altair as alt
from altair.datasets import data
from pathlib import Path

# Chart 1

In [3]:
DATA_PATH = Path("../data/pppub26.csv")
OUTPUT_DIR = Path("../milestones")

variables = [
    "A_AGE",       # Age
    "A_SEX",       # Sex
    "A_HGA",       # Education
    "PEHSPNON",    # Hispanic origin
    "PRDTRACE",    # Race
    "A_FNLWGT",    # Survey weight
    "PEARNVAL",    # Earnings
    "WSAL_VAL",    # Wage and salary income
    "PTOTVAL",     # Total personal income
    "OCCUP",       # Occupation
]

df = pd.read_csv(DATA_PATH, usecols=variables)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (134729, 10)


,PEHSPNON,PRDTRACE,A_AGE,A_SEX,A_FNLWGT,A_HGA,OCCUP,PEARNVAL,PTOTVAL,WSAL_VAL
0,2,1,57,2,112419,39,0,0,20208,0
1,2,1,67,1,98373,39,0,0,42900,0
2,2,1,71,1,88828,39,0,0,27538,0
3,2,1,78,1,124227,44,1006,30000,65821,30000
4,2,1,68,1,124145,39,0,0,14230,0


In [4]:
print("Earnings summary:")
display(df["PEARNVAL"].describe())

print("\nNumber of people with positive earnings:")
print((df["PEARNVAL"] > 0).sum())

print("\nMissing values:")
display(df.isna().sum())

Earnings summary:


count    1.347290e+05
mean     3.830935e+04
std      8.107586e+04
min     -9.999000e+03
25%      0.000000e+00
50%      7.800000e+02
75%      5.400000e+04
max      3.225000e+06
Name: PEARNVAL, dtype: float64


Number of people with positive earnings:
67905

Missing values:


PEHSPNON    0
PRDTRACE    0
A_AGE       0
A_SEX       0
A_FNLWGT    0
A_HGA       0
OCCUP       0
PEARNVAL    0
PTOTVAL     0
WSAL_VAL    0
dtype: int64

In [5]:
for col in ["A_SEX", "PEHSPNON", "PRDTRACE", "OCCUP"]:
    print(f"\n{col}:")
    display(df[col].value_counts().head(15))


A_SEX:


A_SEX
2    68815
1    65914
Name: count, dtype: int64


PEHSPNON:


PEHSPNON
2    106315
1     28414
Name: count, dtype: int64


PRDTRACE:


PRDTRACE
1     102425
2      15309
4       9516
3       1962
6       1449
7       1263
8       1028
5        738
9        200
21       160
15       155
10       147
16       115
11        77
17        54
Name: count, dtype: int64


OCCUP:


OCCUP
0       66679
440      2462
9130     1631
2310     1436
3255     1434
4700     1249
4720     1135
4760     1135
4220     1030
5240     1029
6260      963
9620      942
4020      910
1021      901
4110      784
Name: count, dtype: int64

In [6]:
# 1. Filter people with positive earnings
positive_income = df[df['PEARNVAL'] > 0].copy()

# 2. Define earnings categories
bins = [0, 10000, 20000, 30000, 40000, 50000, 60000, 70000, 80000, 90000, 100000, np.inf]
labels = [
    '$0-$10k', '$10k-$20k', '$20k-$30k', '$30k-$40k', '$40k-$50k',
    '$50k-$60k', '$60k-$70k', '$70k-$80k', '$80k-$90k', '$90k-$100k', '$100k-'
]

# 3. Categorize earnings
positive_income['income_category'] = pd.cut(
    positive_income['PEARNVAL'], bins=bins, labels=labels, right=False
)

display(positive_income[['PEARNVAL', 'income_category']].head(15))

,PEARNVAL,income_category
3,30000,$30k-$40k
5,46800,$40k-$50k
8,60000,$60k-$70k
9,70000,$70k-$80k
12,22000,$20k-$30k
14,88000,$80k-$90k
15,90000,$90k-$100k
16,139000,$100k-
17,157000,$100k-
25,70000,$70k-$80k


In [7]:
income_distribution = (
    positive_income
    .groupby('income_category', observed=True)['A_FNLWGT']
    .sum()
    .div(100)
    .reset_index(name='weighted_population')
)

display(income_distribution)

,income_category,weighted_population
0,$0-$10k,13907553.99
1,$10k-$20k,12886033.32
2,$20k-$30k,16382442.22
3,$30k-$40k,17663501.52
4,$40k-$50k,17580229.53
5,$50k-$60k,16794253.25
6,$60k-$70k,13976170.46
7,$70k-$80k,11138095.03
8,$80k-$90k,8977240.07
9,$90k-$100k,6554243.89


In [8]:
chart01 = alt.Chart(income_distribution).mark_bar(
    color="#C2C0BF"
).encode(
    x=alt.X('income_category:N', sort=labels,title='Annual earnings'),
    y=alt.Y('weighted_population:Q', title ='Estimated Number of Earners')
).properties(
    width=650,
    height=350,
    title='Distribution of Annual Earnings in the United States'
)

rule = alt.Chart(df).mark_rule(color='red').encode(
    y='mean(df[positive_income]):Q'
)

chart01

alt.Chart(...)

In [9]:
chart01.save("../img/01.png", scale_factor=2)

# Chart 2

In [10]:
conditions = [
    (df["PEHSPNON"] == 1) & (df["A_SEX"] == 1),
    (df["PEHSPNON"] == 1) & (df["A_SEX"] == 2),
    (df["PRDTRACE"] == 1) & (df["PEHSPNON"] == 2) & (df["A_SEX"] == 1),
    (df["PRDTRACE"] == 1) & (df["PEHSPNON"] == 2) & (df["A_SEX"] == 2),
    (df["PRDTRACE"] == 2) & (df["PEHSPNON"] == 2) & (df["A_SEX"] == 1),
    (df["PRDTRACE"] == 2) & (df["PEHSPNON"] == 2) & (df["A_SEX"] == 2),
    (df["PRDTRACE"] == 4) & (df["PEHSPNON"] == 2) & (df["A_SEX"] == 1),
    (df["PRDTRACE"] == 4) & (df["PEHSPNON"] == 2) & (df["A_SEX"] == 2)
]

choices = [
    "Hispanic/male",
    "Hispanic/female",
    "White/male",
    "White/female",
    "Black/male",
    "Black/female",
    "Asian/male",
    "Asian/female"
]

df["race_group"] = np.select(
    conditions,
    choices,
    default="Other/Multiracial"
)

display(df["race_group"].value_counts())

race_group
White/female         38991
White/male           37965
Hispanic/female      14336
Hispanic/male        14078
Black/female          7579
Black/male            6518
Other/Multiracial     5973
Asian/female          4874
Asian/male            4415
Name: count, dtype: int64

In [11]:
print(df.columns.tolist())
print(positive_income.columns.tolist())

['PEHSPNON', 'PRDTRACE', 'A_AGE', 'A_SEX', 'A_FNLWGT', 'A_HGA', 'OCCUP', 'PEARNVAL', 'PTOTVAL', 'WSAL_VAL', 'race_group']
['PEHSPNON', 'PRDTRACE', 'A_AGE', 'A_SEX', 'A_FNLWGT', 'A_HGA', 'OCCUP', 'PEARNVAL', 'PTOTVAL', 'WSAL_VAL', 'income_category']


In [12]:
race_income_distribution = (
    positive_income
    .groupby(['income_category', 'race_group'], observed=True)["A_FNLWGT"]
    .sum()
    .div(100)
    .reset_index(name="estimated_population")
)

display(race_income_distribution.head(20))

KeyError: 'race_group'

In [ ]:
race_income_distribution["percentage"] = (
    race_income_distribution["estimated_population"]
    / race_income_distribution.groupby("race_group")[
        "estimated_population"
    ].transform("sum")
    * 100
)
print(race_income_distribution["percentage"])

NameError: name 'race_income_distribution' is not defined

In [ ]:
group_order = [
    "White/male", "White/female",
    "Black/male", "Black/female",
    "Hispanic/male", "Hispanic/female",
    "Asian/male", "Asian/female",
    "Other/Multiracial"
]

group_colors = [
    "#723C91", "#B45FE6",
    "#1EBA1B", "#26EC22",
    "#C46716", "#FA8825",
    "#A51A0D", "#EF2310",
    "#A2A2A2"
]

In [ ]:
chart02 = alt.Chart(race_income_distribution).mark_line(
    point=True
).encode(
    x=alt.X(
        'income_category:N',
        sort=labels,
        title='Annual earnings'
    ),
    y=alt.Y(
        "percentage:Q",
        title="Share of earners within group (%)"
    ),
    color=alt.Color(
        "race_group:N",
        scale=alt.Scale(
            domain=group_order,
            range=group_colors
        ),
        legend=alt.Legend(title="Race / Sex")
    )
).properties(
    width=700,
    height=400,
    title='Earnings Distribution by Race and Sex'
)

chart02

alt.Chart(...)

In [ ]:
chart02.save("../img/02.png", scale_factor=2)

# Chart 3

In [ ]:
# box_data = positive_income[
#     positive_income["race_group"] != "Other/Multiracial"
# ].copy()

In [ ]:
box_summary = (
    box_data.groupby("race_group")
    .apply(
        lambda g: pd.Series({
            "p25": weighted_percentile(
                g["PEARNVAL"], g["A_FNLWGT"], 25
            ),
            "median": weighted_percentile(
                g["PEARNVAL"], g["A_FNLWGT"], 50
            ),
            "p75": weighted_percentile(
                g["PEARNVAL"], g["A_FNLWGT"], 75
            )
        }),
        include_groups=False
    )
    .reset_index()
)

display(box_summary)

,race_group,p25,median,p75
0,Asian/female,34000,63000,110000
1,Asian/male,40000,85000,150000
2,Black/female,25000,45000,68000
3,Black/male,30000,50000,80000
4,Hispanic/female,20500,37500,60000
5,Hispanic/male,30000,48000,75000
6,White/female,25000,50000,80000
7,White/male,38000,70000,115000


In [ ]:
# group_order = [
#     "White/male", "White/female",
#     "Black/male", "Black/female",
#     "Hispanic/male", "Hispanic/female",
#     "Asian/male", "Asian/female"
# ]

# group_colors = [
#     "#723C91", "#B45FE6",
#     "#1EBA1B", "#26EC22",
#     "#C46716", "#FA8825",
#     "#A51A0D", "#EF2310"
# ]

In [ ]:
base = alt.Chart(box_summary).encode(
    y=alt.Y(
        "race_group:N",
        sort=group_order,
        title="Race / Gender"
    )
)

boxes = base.mark_bar(size=22).encode(
    x=alt.X(
        "p25:Q",
        title="Annual earnings ($)",
        axis=alt.Axis(format="$,.0f")
    ),
    x2=alt.X2("p75:Q"),
    color=alt.Color(
        "race_group:N",
        scale=alt.Scale(
            domain=group_order,
            range=group_colors
        ),
        legend=None
    )
)

medians = base.mark_tick(
    color="white",
    thickness=3,
    size=22
).encode(
    x="median:Q"
)

chart03 = (boxes + medians).properties(
    width=650,
    height=350,
    title="Annual Earnings by Race and Gender: Median and IQR"
)

chart03

alt.LayerChart(...)

In [ ]:
chart03.save("../img/03.png", scale_factor=2)

# Chart 3b

In [ ]:
def weighted_percentile(values, weights, percentile):
    order = np.argsort(values)
    values = np.asarray(values)[order]
    weights = np.asarray(weights)[order]

    cumulative_weights = np.cumsum(weights)
    cutoff = weights.sum() * percentile / 100

    return values[
        np.searchsorted(cumulative_weights, cutoff)
    ]

In [ ]:
# Chart 3b: Combine male and female earnings by race

box_data2 = df[
    (df["PEARNVAL"] > 0) &
    (df["race_group2"] != "Other/Multiracial")
].copy()

box_summary2 = (
    box_data2.groupby("race_group2")
    .apply(
        lambda g: pd.Series({
            "p25": weighted_percentile(
                g["PEARNVAL"], g["A_FNLWGT"], 25
            ),
            "median": weighted_percentile(
                g["PEARNVAL"], g["A_FNLWGT"], 50
            ),
            "p75": weighted_percentile(
                g["PEARNVAL"], g["A_FNLWGT"], 75
            )
        }),
        include_groups=False
    )
    .reset_index()
)

group_order2 = ["White", "Black", "Hispanic", "Asian"]
group_colors2 = ["#723C91", "#1EBA1B", "#FA8825", "#A51A0D"]

base2 = alt.Chart(box_summary2).encode(
    y=alt.Y(
        "race_group2:N",
        sort=group_order2,
        title="Race / Ethnicity"
    )
)

boxes2 = base2.mark_bar(size=22).encode(
    x=alt.X(
        "p25:Q",
        title="Annual earnings ($)",
        axis=alt.Axis(format="$,.0f")
    ),
    x2="p75:Q",
    color=alt.Color(
        "race_group2:N",
        scale=alt.Scale(
            domain=group_order2,
            range=group_colors2
        ),
        legend=None
    )
)

medians2 = base2.mark_tick(
    color="white",
    thickness=3,
    size=22
).encode(
    x="median:Q"
)

chart03b = (boxes2 + medians2).properties(
    width=650,
    height=250,
    title="Annual Earnings by Race/Ethnicity: Median and IQR"
)

chart03b

alt.LayerChart(...)

In [ ]:
box_summary2["median"] = earnings_data.groupby("race_group2").apply(
    lambda g: weighted_percentile(
        g["PEARNVAL"], g["A_FNLWGT"], 50
    ),
    include_groups=False
).reindex(box_summary2["race_group2"]).to_numpy()

In [ ]:
# Chart 3b: Median annual earnings by race/ethnicity

group_order2 = ["White", "Black", "Hispanic", "Asian"]
group_colors2 = ["#723C91", "#1EBA1B", "#FA8825", "#A51A0D"]

# Create dollar labels
box_summary2["median_label"] = box_summary2["median"].map(
    lambda x: f"${x:,.0f}"
)

base2 = alt.Chart(box_summary2).encode(
    y=alt.Y(
        "race_group2:N",
        sort=group_order2,
        title="Race / Ethnicity"
    )
)

# IQR bars
boxes2 = base2.mark_bar(size=22).encode(
    x=alt.X(
        "p25:Q",
        title="Annual earnings ($)",
        axis=alt.Axis(format="$,.0f")
    ),
    x2="p75:Q",
    color=alt.Color(
        "race_group2:N",
        scale=alt.Scale(
            domain=group_order2,
            range=group_colors2
        ),
        legend=None
    )
)

# White median markers
medians2 = base2.mark_tick(
    color="white",
    thickness=3,
    size=22
).encode(
    x="median:Q"
)

# Dollar labels above median markers
median_labels2 = base2.mark_text(
    dy=-18,
    fontSize=13,
    fontWeight="bold",
    color="black"
).encode(
    x="median:Q",
    text="median_label:N"
)

chart03b = (
    boxes2 + medians2 + median_labels2
).properties(
    width=650,
    height=250,
    title="Median Annual Earnings by Race/Ethnicity (2025)"
)

chart03b


alt.LayerChart(...)

In [ ]:
chart03b.save("../img/03b.svg")

In [ ]:
print(
    earnings_data.groupby("race_group2")
    .apply(
        lambda g: pd.Series({
            "Mean": np.average(
                g["PEARNVAL"],
                weights=g["A_FNLWGT"]
            ),
            "Median": weighted_percentile(
                g["PEARNVAL"], g["A_FNLWGT"], 50
            )
        }),
        include_groups=False
    ).round(0)
)

                 Mean   Median
race_group2                   
Asian        101871.0  72000.0
Black         63559.0  48000.0
Hispanic      58241.0  42000.0
White         81930.0  60000.0


In [ ]:
# Chart 3b: Mean annual earnings by race/ethnicity
# Source: 2026 CPS ASEC (earnings reference year: 2025)


# 1. Load CPS data
cps_df = pd.read_csv(
    "../data/pppub26.csv",
    usecols=[
        "PEARNVAL",
        "A_FNLWGT",
        "PEHSPNON",
        "PRDTRACE"
    ]
)

# 2. Create mutually exclusive race/ethnicity groups
conditions = [
    cps_df["PEHSPNON"] == 1,
    (cps_df["PRDTRACE"] == 1) & (cps_df["PEHSPNON"] == 2),
    (cps_df["PRDTRACE"] == 2) & (cps_df["PEHSPNON"] == 2),
    (cps_df["PRDTRACE"] == 4) & (cps_df["PEHSPNON"] == 2)
]

choices = ["Hispanic", "White", "Black", "Asian"]

cps_df["race_group2"] = np.select(
    conditions,
    choices,
    default="Other/Multiracial"
)

# 3. Filter to people with positive earnings and weights
earnings_data = cps_df[
    (cps_df["PEARNVAL"] > 0) &
    (cps_df["A_FNLWGT"] > 0) &
    (cps_df["race_group2"] != "Other/Multiracial")
].copy()

# 4. Weighted percentile function
def weighted_percentile(values, weights, percentile):
    order = np.argsort(values)
    values = np.asarray(values)[order]
    weights = np.asarray(weights)[order]

    cutoff = weights.sum() * percentile / 100
    index = np.searchsorted(np.cumsum(weights), cutoff)

    return values[min(index, len(values) - 1)]

# 5. Calculate weighted mean and IQR
box_summary2 = (
    earnings_data.groupby("race_group2")
    .apply(
        lambda g: pd.Series({
            "p25": weighted_percentile(
                g["PEARNVAL"], g["A_FNLWGT"], 25
            ),
            "mean": np.average(
                g["PEARNVAL"],
                weights=g["A_FNLWGT"]
            ),
            "p75": weighted_percentile(
                g["PEARNVAL"], g["A_FNLWGT"], 75
            )
        }),
        include_groups=False
    )
    .reset_index()
)

box_summary2["mean_label"] = box_summary2["mean"].map(
    lambda x: f"${x:,.0f}"
)

display(box_summary2)

# 6. Chart settings
group_order2 = ["White", "Black", "Hispanic", "Asian"]
group_colors2 = ["#723C91", "#1EBA1B", "#FA8825", "#A51A0D"]

base2 = alt.Chart(box_summary2).encode(
    y=alt.Y(
        "race_group2:N",
        sort=group_order2,
        title="Race / Ethnicity"
    )
)

# IQR bars (25th to 75th percentile)
boxes2 = base2.mark_bar(size=22).encode(
    x=alt.X(
        "p25:Q",
        title="Annual earnings ($)",
        axis=alt.Axis(format="$,.0f")
    ),
    x2="p75:Q",
    color=alt.Color(
        "race_group2:N",
        scale=alt.Scale(
            domain=group_order2,
            range=group_colors2
        ),
        legend=None
    )
)

# Mean markers
means2 = base2.mark_point(
    filled=True,
    color="black",
    size=100
).encode(
    x="mean:Q"
)

# Dollar labels
labels2 = base2.mark_text(
    dy=-18,
    fontSize=13,
    fontWeight="bold",
    color="black"
).encode(
    x="mean:Q",
    text="mean_label:N"
)

# 7. Combine and export
chart03d = (
    boxes2 + means2 + labels2
).properties(
    width=650,
    height=250,
    title="Mean Annual Earnings by Race/Ethnicity (2025)"
)

chart03d


,race_group2,p25,mean,p75,mean_label
0,Asian,36000.0,101871.295267,130000.0,"$101,871"
1,Black,28000.0,63559.424706,75000.0,"$63,559"
2,Hispanic,25000.0,58240.760446,68000.0,"$58,241"
3,White,30000.0,81930.099800,100000.0,"$81,930"


alt.LayerChart(...)

In [ ]:
chart03d.save("../img/03d.svg")

In [ ]:
conditions = [
    (df["PEHSPNON"] == 1),
    (df["PRDTRACE"] == 1) & (df["PEHSPNON"] == 2),
    (df["PRDTRACE"] == 2) & (df["PEHSPNON"] == 2),
    (df["PRDTRACE"] == 4) & (df["PEHSPNON"] == 2)
]

choices = [
    "Hispanic",
    "White",
    "Black",
    "Asian"
]

df["race_group2"] = np.select(
    conditions,
    choices,
    default="Other/Multiracial"
)

display(df["race_group2"].value_counts())

race_group2
White                76956
Hispanic             28414
Black                14097
Asian                 9289
Other/Multiracial     5973
Name: count, dtype: int64

In [ ]:
box_summary2 = (
    box_data.groupby("race_group2")
    .apply(
        lambda g: pd.Series({
            "p25": weighted_percentile(
                g["PEARNVAL"], g["A_FNLWGT"], 25
            ),
            "median": weighted_percentile(
                g["PEARNVAL"], g["A_FNLWGT"], 50
            ),
            "p75": weighted_percentile(
                g["PEARNVAL"], g["A_FNLWGT"], 75
            )
        }),
        include_groups=False
    )
    .reset_index()
)

display(box_summary2)

NameError: name 'box_data' is not defined

In [ ]:
group_order2 = [
    "White"
    "Black"
    "Hispanic",
    "Asian"
]

group_colors2 = [
    "#723C91",
    "#1EBA1B",
    "#FA8825",
    "#A51A0D"
]

In [ ]:
base = alt.Chart(box_summary).encode(
    y=alt.Y(
        "race_group:N",
        sort=group_order,
        title="Race / Gender"
    )
)

boxes = base.mark_bar(size=22).encode(
    x=alt.X(
        "p25:Q",
        title="Annual earnings ($)",
        axis=alt.Axis(format="$,.0f")
    ),
    x2=alt.X2("p75:Q"),
    color=alt.Color(
        "race_group:N",
        scale=alt.Scale(
            domain=group_order,
            range=group_colors
        ),
        legend=None
    )
)

medians = base.mark_tick(
    color="white",
    thickness=3,
    size=22
).encode(
    x="median:Q"
)

chart03 = (boxes + medians).properties(
    width=650,
    height=350,
    title="Annual Earnings by Race and Gender: Median and IQR"
)

chart03

In [ ]:
population = pd.DataFrame({
    "Group": ["White", "Hispanic", "Black", "Asian", "Other"],
    "Population": [
        191697647, 62080044, 39940338, 19618719, 18112533
    ]
})

population["Percent"] = (
    population["Population"] / population["Population"].sum() * 100
)

population["Label"] = population["Percent"].map(
    lambda x: f"{x:.1f}%"
)

group_order = ["White", "Hispanic", "Black", "Asian", "Other"]

population["sort_order"] = population["Group"].map({
    "White": 0,
    "Hispanic": 1,
    "Black": 2,
    "Asian": 3,
    "Other": 4
})

base = alt.Chart(population).encode(
    theta=alt.Theta("Population:Q", stack=True),
    order=alt.Order("sort_order:Q", sort="ascending"),
    color=alt.Color(
        "Group:N",
        scale=alt.Scale(
            domain=group_order,
            range=["#723C91", "#FA8825", "#1EBA1B",
                   "#A51A0D", "#999999"]
        ),
        legend=alt.Legend(title="Race / Ethnicity")
    )
)

pie = base.mark_arc(outerRadius=120)

labels = base.mark_text(
    radius=145,
    fontSize=13,
    fontWeight="bold"
).encode(text="Label:N")

chart03c = (pie + labels).properties(
    width=340,
    height=340,
    title="U.S. Population by Race/Ethnicity (2020)"
)

chart03c


alt.LayerChart(...)

In [ ]:
chart03c.save("../img/03c.svg")

#### Annual Earnings by Race and Gender
This chart compares survey-weighted annual earnings across eight demographic groups using the 2026 CPS ASEC. The horizontal bars represent the middle 50% of earnings (25th–75th percentiles), while white lines indicate median earnings. Differences in median earnings and earnings dispersion highlight disparities across demographic groups.

# Chart 4

In [ ]:
occupation_counts = (
    positive_income[positive_income["OCCUP"] != 0]
    .groupby("OCCUP")["A_FNLWGT"]
    .sum()
    .div(100)
    .sort_values(ascending=False)
    .head(15)
)

display(occupation_counts)

OCCUP
440     6165258.88
9130    4146930.48
2310    3660696.13
3255    3655586.92
4700    3174467.44
4720    3024148.58
4760    2889701.44
5240    2738193.87
1021    2579381.27
9620    2556200.80
4220    2501117.56
6260    2385268.17
4110    2233050.66
4020    2201222.50
9645    2109308.77
Name: A_FNLWGT, dtype: float64

In [ ]:
occupation_income = (
    positive_income
    .groupby(['income_category', 'OCCUP'], observed=True)["A_FNLWGT"]
    .sum()
    .div(100)
    .reset_index(name="estimated_population")
)

display(occupation_income.head(20))

,income_category,OCCUP,estimated_population
0,$0-$10k,10,31143.91
1,$0-$10k,20,17737.86
2,$0-$10k,40,4655.80
3,$0-$10k,51,39482.27
4,$0-$10k,60,6598.33
5,$0-$10k,110,4773.81
6,$0-$10k,120,5465.70
7,$0-$10k,140,4424.62
8,$0-$10k,160,1362.51
9,$0-$10k,205,81298.22


In [ ]:
occupation_income["percentage"] = (
    occupation_income["estimated_population"]
    / occupation_income.groupby("OCCUP")[
        "estimated_population"
    ].transform("sum")
    * 100
)
print(occupation_income["percentage"])

0        1.578999
1        1.264655
2        9.218553
3        5.801850
4        4.857390
          ...    
4426     1.502863
4427    13.214095
4428     3.194586
4429    28.911647
4430    10.563392
Name: percentage, Length: 4431, dtype: float64


In [ ]:
occupation_codes = pd.read_csv("../data/cps_occupation_codes.csv")

print(occupation_codes.columns.tolist())
display(occupation_codes.head(10))

['ROW_TYPE', '2020 Census Public Use Occupation Code', '2018 SOC Code', 'Description (2020 Census Occupation Code)']


,ROW_TYPE,2020 Census Public Use Occupation Code,2018 SOC Code,Description (2020 Census Occupation Code)
0,intro-range,0010 - 3550,NaN,"Management, Business, Science, and Arts Occupa..."
1,intro-range,3601 - 4655,NaN,Service Occupations
2,intro-range,4700 - 5940,NaN,Sales and Office Occupations
3,intro-range,6005 - 7640,NaN,"Natural Resources, Construction, and Maintenan..."
4,intro-range,7700 - 9760,NaN,"Production, Transportation, and Material Movin..."
5,broad-group,0010-3550,11-0000 - 29-0000,"Management, Business, Science, and Arts Occupa..."
6,broad-group,0010-0960,11-0000 - 13-0000,"Management, Business, and Financial Occupations"
7,group,0010-0440,11-0000,Management Occupations
8,detail,0010,11-10XX,Chief executives
9,detail,0020,11-1021,General and operations managers


In [ ]:
selected_codes = [
    6050,  # Other agricultural workers
    6260,  # Construction laborers
    8990,  # Miscellaneous production workers
    4760,  # Retail salespersons
    2310,  # Elementary and middle school teachers
    3255,  # Registered nurses
    800,   # Accountants and auditors
    845,   # Financial and investment analysts
    1021,  # Software developers
    2100   # Lawyers and judicial workers
]

occupation_sample = positive_income[
    positive_income["OCCUP"].isin(selected_codes)
].copy()

display(
    occupation_sample["OCCUP"]
    .value_counts()
    .sort_index()
)

OCCUP
800      643
845      165
1021     900
2100     519
2310    1436
3255    1434
4760    1134
6050     380
6260     959
8990     475
Name: count, dtype: int64

In [ ]:
selected_occupations = positive_income[
    positive_income["OCCUP"].isin(selected_codes)
].copy()

print(selected_occupations["OCCUP"].value_counts())

OCCUP
2310    1436
3255    1434
4760    1134
6260     959
1021     900
800      643
2100     519
8990     475
6050     380
845      165
Name: count, dtype: int64


In [ ]:
def weighted_median(values, weights):
    order = np.argsort(values)
    values = np.asarray(values)[order]
    weights = np.asarray(weights)[order]

    cumulative_weights = np.cumsum(weights)
    cutoff = weights.sum() / 2

    return values[np.searchsorted(cumulative_weights, cutoff)]

In [ ]:
occupation_medians = (
    selected_occupations
    .groupby("OCCUP")
    .apply(
        lambda group: weighted_median(
            group["PEARNVAL"],
            group["A_FNLWGT"]
        ),
        include_groups=False
    )
    .reset_index(name="median_earnings")
)

display(occupation_medians)

,OCCUP,median_earnings
0,800,78000
1,845,95000
2,1021,130000
3,2100,165000
4,2310,60000
5,3255,79000
6,4760,29000
7,6050,30000
8,6260,40000
9,8990,43600


In [ ]:
occupation_names = {
    6050: "Agricultural workers",
    6260: "Construction laborers",
    8990: "Production workers",
    4760: "Retail salespersons",
    2310: "Elementary/middle school teachers",
    3255: "Registered nurses",
    800: "Accountants and auditors",
    845: "Financial analysts",
    1021: "Software developers",
    2100: "Lawyers and judicial workers"
}

occupation_medians["occupation"] = (
    occupation_medians["OCCUP"].map(occupation_names)
)

In [ ]:
# occupation_ranges = (
#     selected_occupations
#     .groupby("OCCUP")["PEARNVAL"]
#     .agg(min_earnings="min", max_earnings="max")
#     .reset_index()
# )

# occupation_medians = occupation_medians.merge(
#     occupation_ranges,
#     on="OCCUP"
# )

# display(occupation_medians)

In [ ]:
def weighted_percentile(values, weights, percentile):
    order = np.argsort(values)
    values = np.asarray(values)[order]
    weights = np.asarray(weights)[order]

    cumulative_weights = np.cumsum(weights)
    cutoff = weights.sum() * percentile / 100

    return values[np.searchsorted(cumulative_weights, cutoff)]

In [ ]:
occupation_percentiles = (
    selected_occupations
    .groupby("OCCUP")
    .apply(
        lambda group: pd.Series({
            "p25": weighted_percentile(
                group["PEARNVAL"], group["A_FNLWGT"], 25
            ),
            "p75": weighted_percentile(
                group["PEARNVAL"], group["A_FNLWGT"], 75
            )
        }),
        include_groups=False
    )
    .reset_index()
)

display(occupation_percentiles)

,OCCUP,p25,p75
0,800,55000,105002
1,845,75000,150000
2,1021,95000,180000
3,2100,102000,250000
4,2310,48000,80000
5,3255,58000,101000
6,4760,12000,65000
7,6050,12800,46001
8,6260,25000,60000
9,8990,30000,59000


In [ ]:
# Remove any existing percentile columns
occupation_medians = occupation_medians.drop(
    columns=[
        col for col in occupation_medians.columns
        if col.startswith(("p25", "p75"))
    ]
)

# Merge the correct percentiles
occupation_medians = occupation_medians.merge(
    occupation_percentiles,
    on="OCCUP",
    how="left"
)

# Verify
display(
    occupation_medians[
        ["occupation", "p25", "median_earnings", "p75"]
    ]
)

,occupation,p25,median_earnings,p75
0,Accountants and auditors,55000,78000,105002
1,Financial analysts,75000,95000,150000
2,Software developers,95000,130000,180000
3,Lawyers and judicial workers,102000,165000,250000
4,Elementary/middle school teachers,48000,60000,80000
5,Registered nurses,58000,79000,101000
6,Retail salespersons,12000,29000,65000
7,Agricultural workers,12800,30000,46001
8,Construction laborers,25000,40000,60000
9,Production workers,30000,43600,59000


In [ ]:
base = alt.Chart(occupation_medians).encode(
    y=alt.Y(
        "occupation:N",
        sort=alt.SortField(
            field="median_earnings",
            order="descending"
        ),
        title=None
    )
)

ranges = base.mark_rule(
    color="#A0A0A0",
    strokeWidth=5
).encode(
    x=alt.X(
        "p25:Q",
        title="Annual earnings ($)",
        axis=alt.Axis(format="$,.0f")
    ),
    x2="p75:Q"
)

dots = base.mark_circle(
    size=110,
    color="#E04F2B"
).encode(
    x="median_earnings:Q",
    tooltip=[
        "occupation:N",
        alt.Tooltip("p25:Q", title="25th percentile", format="$,.0f"),
        alt.Tooltip("median_earnings:Q", title="Median", format="$,.0f"),
        alt.Tooltip("p75:Q", title="75th percentile", format="$,.0f")
    ]
)

chart04 = (ranges + dots).properties(
    width=650,
    height=350,
    title="Median and Interquartile Range of Annual Earnings by Occupation"
)

chart04

alt.LayerChart(...)

In [ ]:
chart04.save("../img/04.png", scale_factor=2)

#### Median Annual Earnings by Occupation
This chart compares survey-weighted median annual earnings across ten selected occupations using the 2026 CPS ASEC. The dots represent median earnings, while the bars show the 25th–75th percentile range. The differences highlight substantial earnings disparities across occupations. Estimates include workers with positive annual earnings, regardless of hours worked.

# Chart 5

In [ ]:
income_cols = [
    "A_FNLWGT",
    "PTOTVAL",
    "WSAL_VAL",
    "SEMP_VAL",
    "FRSE_VAL",
    "INT_VAL",
    "DIV_VAL",
    "RNT_VAL"
]

income_df = pd.read_csv(
    "../data/pppub26.csv",
    usecols=income_cols
)

income_df.head()

,A_FNLWGT,DIV_VAL,FRSE_VAL,INT_VAL,PTOTVAL,RNT_VAL,SEMP_VAL,WSAL_VAL
0,112419,0,0,0,20208,0,0,0
1,98373,0,0,0,42900,0,0,0
2,88828,0,0,2,27538,0,0,0
3,124227,0,0,101,65821,20,0,30000
4,124145,0,0,10,14230,0,0,0


In [ ]:
print(income_df.shape)
print(income_df.isna().sum())
print(income_df.describe())

(134729, 8)
A_FNLWGT    0
DIV_VAL     0
FRSE_VAL    0
INT_VAL     0
PTOTVAL     0
RNT_VAL     0
SEMP_VAL    0
WSAL_VAL    0
dtype: int64
           A_FNLWGT        DIV_VAL      FRSE_VAL        INT_VAL       PTOTVAL  \
count  1.347290e+05  134729.000000  1.347290e+05  134729.000000  1.347290e+05   
mean   2.506029e+05     840.410580  1.899606e+02    2709.333558  5.049569e+04   
std    2.672761e+05   12042.996071  7.766324e+03   12061.883387  8.944104e+04   
min    0.000000e+00       0.000000 -1.999800e+04       0.000000 -9.999000e+03   
25%    0.000000e+00       0.000000  0.000000e+00       0.000000  0.000000e+00   
50%    1.703180e+05       0.000000  0.000000e+00       0.000000  2.601500e+04   
75%    4.501820e+05       0.000000  0.000000e+00     250.000000  6.550100e+04   
max    3.034465e+06  999999.000000  1.072500e+06  249999.000000  3.277551e+06   

             RNT_VAL      SEMP_VAL      WSAL_VAL  
count  134729.000000  1.347290e+05  1.347290e+05  
mean      594.233365  1.789211e

In [ ]:
# Keep people with positive total personal income and positive weights
income_clean = income_df[
    (income_df["PTOTVAL"] > 0) &
    (income_df["A_FNLWGT"] > 0)
].copy()

# Define income groups
income_clean["income_group"] = pd.cut(
    income_clean["PTOTVAL"],
    bins=[0, 30000, 60000, 100000, 300000, 500000, 1000000, float("inf")],
    labels=["Under $30k", "$30k–$60k", "$60k–$100k", "$100k-$300k",
            "$300k-$500k", "$500k-$1m", "$1m-"]
)

# Define income sources (positive amounts only)
income_clean["Wages"] = income_clean["WSAL_VAL"].clip(lower=0)

income_clean["Self-employment"] = (
    income_clean["SEMP_VAL"].clip(lower=0) +
    income_clean["FRSE_VAL"].clip(lower=0)
)

income_clean["Investment income"] = (
    income_clean["INT_VAL"].clip(lower=0) +
    income_clean["DIV_VAL"].clip(lower=0) +
    income_clean["RNT_VAL"].clip(lower=0)
)

In [ ]:
sources = ["Wages", "Self-employment", "Investment income"]

for source in sources:
    income_clean[source] *= income_clean["A_FNLWGT"] / 100

income_summary = (
    income_clean
    .groupby("income_group", observed=True)[sources]
    .sum()
    .reset_index()
)

income_long = income_summary.melt(
    id_vars="income_group",
    var_name="source",
    value_name="amount"
)

income_long["percentage"] = (
    income_long["amount"] /
    income_long.groupby("income_group")["amount"].transform("sum") * 100
)

income_long["stack_order"] = income_long["source"].map({
    "Wages": 0,
    "Self-employment": 1,
    "Investment income": 2
})

income_long.head(12)

,income_group,source,amount,percentage,stack_order
0,Under $30k,Wages,5.552301e+11,85.562206,0
1,$30k–$60k,Wages,2.029313e+12,90.833507,0
2,$60k–$100k,Wages,2.736313e+12,90.063128,0
3,$100k-$300k,Wages,5.037110e+12,84.118842,0
4,$300k-$500k,Wages,9.017879e+11,79.373342,0
5,$500k-$1m,Wages,6.942158e+11,81.778374,0
6,$1m-,Wages,6.679776e+11,85.334787,0
7,Under $30k,Self-employment,4.621405e+10,7.121690,1
8,$30k–$60k,Self-employment,9.871533e+10,4.418569,1
9,$60k–$100k,Self-employment,1.052096e+11,3.462873,1


In [ ]:
chart05 = alt.Chart(income_long).mark_bar().encode(
    x=alt.X(
        "income_group:N",
        title="Annual personal income",
        sort=["Under $30k", "$30k–$60k", "$60k–$100k", "$100k+"]
    ),
    y=alt.Y(
        "percentage:Q",
        title="Share of selected positive income sources (%)",
        scale=alt.Scale(domain=[0, 100])
    ),
    color=alt.Color(
        "source:N",
        scale=alt.Scale(
            domain=["Investment income", "Self-employment", "Wages"],
            range=["#F45314", "#C7E217", "#09AE2C"]
        ),
        legend=alt.Legend(title="Income source")
    ),
    order=alt.Order("stack_order:Q", sort="ascending"),
    tooltip=[
        "income_group:N",
        "source:N",
        alt.Tooltip("percentage:Q", format=".1f", title="Share (%)")
    ]
).properties(
    width=600,
    height=400,
    title="Composition of Income Sources by Income Group"
)

chart05

alt.Chart(...)

In [ ]:
chart05.save("../img/05.png", scale_factor=2)

In [ ]:
import pandas as pd
import altair as alt
from pathlib import Path

# 1. Load data
df = pd.read_csv("../data/SCFP2022.csv")

income_sources = {
    "WAGEINC": "Wages",
    "BUSSEFARMINC": "Business income",
    "INTDIVINC": "Interest & dividends",
    "KGINC": "Capital gains"
}

income_order = [
    "Bottom 50%",
    "Next 40%",
    "Next 9%",
    "Top 1%"
]

# 2. Identify the five imputations per family
df["implicate"] = df.groupby("YY1").cumcount() + 1

# 3. Assign income groups using survey-weighted percentiles
def assign_income_group(data):
    data = data.sort_values("INCOME").copy()

    data["percentile"] = (
        data["WGT"].cumsum() - data["WGT"] / 2
    ) / data["WGT"].sum()

    data["Income group"] = pd.cut(
        data["percentile"],
        bins=[0, 0.50, 0.90, 0.99, 1.0],
        labels=income_order,
        include_lowest=True
    )

    return data

df = pd.concat(
    [assign_income_group(d) for _, d in df.groupby("implicate")],
    ignore_index=True
)

# Divide weights by five to account for the five imputations
df["weight"] = df["WGT"] / 5

# 4. Calculate income shares within each group
rows = []

for group, data in df.groupby("Income group", observed=True):

    total_income = (
        data["INCOME"] * data["weight"]
    ).sum()

    for variable, label in income_sources.items():

        source_income = (
            data[variable] * data["weight"]
        ).sum()

        rows.append({
            "Income group": str(group),
            "Income source": label,
            "Share (%)": source_income / total_income * 100
        })

chart_data = pd.DataFrame(rows)

# 5. Display exact percentages
income_table = chart_data.pivot(
    index="Income group",
    columns="Income source",
    values="Share (%)"
).reindex(income_order)

display(income_table.round(1))

# 6. Create grouped bar chart
source_order = list(income_sources.values())

chart05b = alt.Chart(chart_data).mark_bar().encode(
    x=alt.X(
        "Income group:N",
        sort=income_order,
        title="Family income group",
        axis=alt.Axis(labelAngle=0)
    ),
    xOffset=alt.XOffset(
        "Income source:N",
        sort=source_order
    ),
    y=alt.Y(
        "Share (%):Q",
        title="Share of total family income (%)"
    ),
    color=alt.Color(
        "Income source:N",
        scale=alt.Scale(
            domain=source_order,
            range=["#0E9613", "#9ADF46", "#EE9944", "#A51A0D"]
        ),
        legend=alt.Legend(title="Income source")
    ),
    tooltip=[
        "Income group:N",
        "Income source:N",
        alt.Tooltip("Share (%):Q", format=".1f")
    ]
).properties(
    width=750,
    height=400,
    title="Where Does the Money Come From? (2022 SCF)"
)

chart05b


/var/folders/3w/plrcmbn551b7m7src59wh39w0000gn/T/ipykernel_1137/1009556543.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["implicate"] = df.groupby("YY1").cumcount() + 1


Income source,Business income,Capital gains,Interest & dividends,Wages
Income group,,,,
Bottom 50%,6.0,0.7,1.1,52.8
Next 40%,8.0,1.8,1.4,72.8
Next 9%,13.5,9.5,4.9,62.9
Top 1%,25.1,30.5,6.0,35.1


alt.Chart(...)

In [ ]:
import pandas as pd
import altair as alt

income_order = [
    "Bottom 50%", "Middle 40%", "Next 9%", "Top 1%"
]

category_order = [
    "Wages",
    "Business & investment",
    "Other income"
]

category_colors = [
    "#0E9613",
    "#A51A0D",
    "#D9D9D9"
]

# 1. Prepare donut data from your existing chart_data
donut_data = chart_data.copy()

donut_data["Category"] = donut_data["Income source"].replace({
    "Wages": "Wages",
    "Business income": "Business & investment",
    "Interest & dividends": "Business & investment",
    "Capital gains": "Business & investment"
})

donut_data = (
    donut_data.groupby(
        ["Income group", "Category"],
        as_index=False
    )["Share (%)"].sum()
)

# Remaining income (e.g. retirement and transfers)
remaining = (
    donut_data.groupby("Income group", as_index=False)["Share (%)"]
    .sum()
)

remaining["Share (%)"] = 100 - remaining["Share (%)"]
remaining["Category"] = "Other income"

donut_data = pd.concat(
    [donut_data, remaining],
    ignore_index=True
)

# Consistent stacking order
donut_data["sort_order"] = donut_data["Category"].map({
    category: i for i, category in enumerate(category_order)
})

donut_data["label"] = donut_data["Share (%)"].map(
    lambda x: f"{x:.1f}%"
)

# 2. Create one donut
def make_donut(group):

    data = donut_data[
        donut_data["Income group"] == group
    ]

    base = alt.Chart(data).encode(
        theta=alt.Theta("Share (%):Q", stack=True),
        order=alt.Order("sort_order:Q", sort="ascending")
    )

    donut = base.mark_arc(
        innerRadius=85,
        outerRadius=125
    ).encode(
        color=alt.Color(
            "Category:N",
            scale=alt.Scale(
                domain=category_order,
                range=category_colors
            ),
            legend=alt.Legend(title="Income source")
        ),
        tooltip=[
            "Category:N",
            alt.Tooltip("Share (%):Q", format=".1f")
        ]
    )

    # Percentage labels around the outside of each segment
    percentages = base.mark_text(
        radius=145,
        fontSize=12,
        fontWeight="bold",
        color="#333333"
    ).encode(
        text="label:N"
    )

    # Group name in the center
    center = alt.Chart(
        pd.DataFrame({"group": [group]})
    ).mark_text(
        fontSize=18,
        fontWeight="bold",
        color="#222222"
    ).encode(
        text="group:N"
    )

    return (donut + percentages + center).properties(
        width=340,
        height=340
    )

# 3. Arrange four charts in a 2x2 layout
chart_donuts = alt.vconcat(
    alt.hconcat(
        make_donut("Bottom 50%"),
        make_donut("Next 40%"),
        spacing=10
    ),
    alt.hconcat(
        make_donut("Next 9%"),
        make_donut("Top 1%"),
        spacing=10
    ),
    spacing=10
).resolve_scale(color="shared")

chart_donuts


alt.VConcatChart(...)

In [ ]:
import pandas as pd
import altair as alt

# 1. Income group names and order
income_order = [
    "Bottom 50%",
    "Middle 40%",
    "Next 9%",
    "Top 1%"
]

# 2. Categories in CLOCKWISE order
category_order = [
    "Wages",
    "Other",
    "Business",
    "Investment"
]

category_colors = [
    "#0E9613",  # Wages - green
    "#D9D9D9",  # Other - gray
    "#EE9944",  # Business - orange
    "#A51A0D"   # Investment - dark red
]

# 3. Prepare data
donut_data = chart_data.copy()

# Rename the income group
donut_data["Income group"] = donut_data["Income group"].replace({
    "Next 40%": "Middle 40%"
})

# Combine interest/dividends and capital gains
donut_data["Category"] = donut_data["Income source"].replace({
    "Wages": "Wages",
    "Business income": "Business",
    "Interest & dividends": "Investment",
    "Capital gains": "Investment"
})

donut_data = (
    donut_data.groupby(
        ["Income group", "Category"],
        as_index=False
    )["Share (%)"].sum()
)

# 4. Calculate Other income
remaining = (
    donut_data.groupby(
        "Income group", as_index=False
    )["Share (%)"].sum()
)

remaining["Share (%)"] = 100 - remaining["Share (%)"]
remaining["Category"] = "Other"

donut_data = pd.concat(
    [donut_data, remaining],
    ignore_index=True
)

# 5. Set clockwise stacking order
donut_data["sort_order"] = donut_data["Category"].map({
    category: i
    for i, category in enumerate(category_order)
})

# Create percentage labels
donut_data["label"] = donut_data["Share (%)"].map(
    lambda x: f"{x:.1f}%"
)

# 6. Create donut function
def make_donut(group):

    data = donut_data[
        donut_data["Income group"] == group
    ].copy()

    base = alt.Chart(data).encode(
        theta=alt.Theta(
            "Share (%):Q",
            stack=True
        ),
        order=alt.Order(
            "sort_order:Q",
            sort="ascending"
        )
    )

    # Donut
    donut = base.mark_arc(
        innerRadius=85,
        outerRadius=125
    ).encode(
        color=alt.Color(
            "Category:N",
            scale=alt.Scale(
                domain=category_order,
                range=category_colors
            ),
            legend=alt.Legend(
                title="Income source",
                values=category_order
            )
        ),
        tooltip=[
            "Income group:N",
            "Category:N",
            alt.Tooltip(
                "Share (%):Q",
                format=".1f",
                title="Share (%)"
            )
        ]
    )

    # Percentage labels outside the ring
    percentages = base.mark_text(
        radius=145,
        fontSize=12,
        fontWeight="bold",
        color="#333333"
    ).encode(
        text="label:N"
    )

    # Income group name in the center
    center = alt.Chart(
        pd.DataFrame({"group": [group]})
    ).mark_text(
        fontSize=18,
        fontWeight="bold",
        color="#222222"
    ).encode(
        text="group:N"
    )

    return (
        donut + percentages + center
    ).properties(
        width=340,
        height=340
    )

# 7. Arrange charts in 2 x 2 layout
chart_donuts = alt.vconcat(
    alt.hconcat(
        make_donut("Bottom 50%"),
        make_donut("Middle 40%"),
        spacing=10
    ),
    alt.hconcat(
        make_donut("Next 9%"),
        make_donut("Top 1%"),
        spacing=10
    ),
    spacing=10
).resolve_scale(
    color="shared"
)

chart_donuts

alt.VConcatChart(...)

In [ ]:
# Export for Illustrator
chart_donuts.save("../img/05b.svg")

# Chart 6

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH2 = Path("../data/dfa-networth-levels.csv")

variables = ["Date", "Category", "Net worth"]

df_wealth = pd.read_csv(DATA_PATH2, usecols=variables)

df_wealth["Year"] = df_wealth["Date"].str[:4].astype(int)

df_wealth["Wealth group"] = df_wealth["Category"].replace({
    "TopPt1": "Top 1%",
    "RemainingTop1": "Top 1%",
    "Next9": "Next 9%",
    "Next40": "Middle 40%",
    "Bottom50": "Bottom 50%"
})

df_wealth["Wealth (trillions)"] = df_wealth["Net worth"] / 1_000_000

df_wealth_selected = df_wealth[
    (df_wealth["Date"] == "1990:Q3") |
    (
        (df_wealth["Year"] % 5 == 0) &
        (df_wealth["Date"].str.endswith("Q3"))
    )
].copy()

wealth_summary = (
    df_wealth_selected
    .groupby(["Date", "Wealth group"], as_index=False)["Wealth (trillions)"]
    .sum()
)

wealth_summary.head()

,Date,Wealth group,Wealth (trillions)
0,1990:Q3,Bottom 50%,0.744277
1,1990:Q3,Middle 40%,7.685969
2,1990:Q3,Next 9%,7.858272
3,1990:Q3,Top 1%,4.721261
4,1995:Q3,Bottom 50%,1.024037


In [ ]:
# DATA_PATH2 = Path("../data/dfa-networth-levels.csv")

# variables = [
# 	"Date",
#     "Category",
#     "Net worth"
# ]

# df_wealth = pd.read_csv(DATA_PATH2, usecols=variables)

In [ ]:
df_wealth["Year"] = df_wealth["Date"].str[:4].astype(int)

# Combine the two top-1% categories
df_wealth["Wealth group"] = df_wealth["Category"].replace({
    "TopPt1": "Top 1%",
    "RemainingTop1": "Top 1%",
    "Next9": "Next 9%",
    "Next40": "Middle 40%",
    "Bottom50": "Bottom 50%"
})

# Federal Reserve figures are in millions of dollars
df_wealth["Wealth (trillions)"] = df_wealth["Net worth"] / 1_000_000

# Keep Q3 for every fifth year, starting in 1990
# 1989 Q3 is the earliest observation, so include it separately
df_wealth_selected = df_wealth[
    (df_wealth["Date"] == "1990:Q3") |
    (
        (df_wealth["Year"] % 5 == 0) &
        (df_wealth["Date"].str.endswith("Q3"))
    )
].copy()

wealth_summary = (
    df_wealth_selected
    .groupby(["Date", "Wealth group"], as_index=False)["Wealth (trillions)"]
    .sum()
)
wealth_summary["stack_order"] = wealth_summary["Wealth group"].map({
    "Top 1%": 0,
    "Next 9%": 1,
    "Middle 40%": 2,
    "Bottom 50%": 3
})

wealth_order = [
    "Bottom 50%",
    "Middle 40%",
    "Next 9%",
    "Top 1%"
]

legend_order = [
    "Top 1%",
    "Next 9%",
    "Middle 40%",
    "Bottom 50%"
]

wealth_summary.head(10)

,Date,Wealth group,Wealth (trillions),stack_order
0,1990:Q3,Bottom 50%,0.744277,3
1,1990:Q3,Middle 40%,7.685969,2
2,1990:Q3,Next 9%,7.858272,1
3,1990:Q3,Top 1%,4.721261,0
4,1995:Q3,Bottom 50%,1.024037,3
5,1995:Q3,Middle 40%,9.787954,2
6,1995:Q3,Next 9%,9.455381,1
7,1995:Q3,Top 1%,7.619192,0
8,2000:Q3,Bottom 50%,1.377520,3
9,2000:Q3,Middle 40%,14.322877,2


In [ ]:
wealth_order = [
    "Bottom 50%",
    "Middle 40%",
    "Next 9%",
    "Top 1%"
]

chart06a = alt.Chart(wealth_summary).mark_bar().encode(
    x=alt.X(
        "Date:N",
        title="Year",
        sort=alt.SortField("Date", order="ascending"),
        axis=alt.Axis(labelAngle=0)
    ),
    y=alt.Y(
        "Wealth (trillions):Q",
        title="Household net worth ($ trillions)"
    ),
    color=alt.Color(
        "Wealth group:N",
        scale=alt.Scale(
            domain=wealth_order,
            range=["#0E9613", "#9ADF46", "#EE9944", "#A51A0D"]
        ),
        legend=alt.Legend(
        title="Wealth group",
        values=legend_order
        )
    ),
    order=alt.Order("stack_order:Q", sort="descending"),
    tooltip=[
        "Date:N",
        "Wealth group:N",
        alt.Tooltip("Wealth (trillions):Q", format=".2f")
    ]
).properties(
    width=750,
    height=400,
    title="Distribution of U.S. Household Wealth Over Time"
)

chart06a

alt.Chart(...)

In [ ]:

wealth_order = [
    "Bottom 50%",
    "Middle 40%",
    "Next 9%",
    "Top 1%"
]

chart06a = alt.Chart(wealth_summary).mark_line(
    point=True,
    strokeWidth=3
).encode(
    x=alt.X(
        "Date:O",
        title="Year",
        sort="ascending",
        axis=alt.Axis(labelAngle=0)
    ),
    y=alt.Y(
        "Wealth (trillions):Q",
        title="Household net worth ($ trillions)",
        scale=alt.Scale(zero=True)
    ),
    color=alt.Color(
        "Wealth group:N",
        scale=alt.Scale(
            domain=wealth_order,
            range=["#0E9613", "#9ADF46", "#EE9944", "#A51A0D"]
        ),
        legend=alt.Legend(
            title="Wealth group",
            values=wealth_order
        )
    ),
    tooltip=[
        "Date:O",
        "Wealth group:N",
        alt.Tooltip(
            "Wealth (trillions):Q",
            format=".2f",
            title="Wealth ($ trillions)"
        )
    ]
).properties(
    width=750,
    height=400,
    title="U.S. Household Wealth by Wealth Group Over Time"
)

chart06a


alt.Chart(...)

In [ ]:
print([name for name in globals() if "wealth" in name.lower()])

['wealth_order', 'df_wealth', 'df_wealth_selected', 'wealth_summary']


In [ ]:
chart06a.save("../img/06a.png", scale_factor=2)

In [ ]:
DATA_PATH3 = Path("../data/dfa-networth-shares.csv")

variables = [
	"Date",
    "Category",
    "Net worth"
]

df_wealth2 = pd.read_csv(DATA_PATH3, usecols=variables)

In [ ]:
df_wealth2["Year"] = df_wealth2["Date"].str[:4].astype(int)

df_wealth2["Wealth group"] = df_wealth2["Category"].replace({
    "TopPt1": "Top 1%",
    "RemainingTop1": "Top 1%",
    "Next9": "Next 9%",
    "Next40": "Middle 40%",
    "Bottom50": "Bottom 50%"
})

df_wealth2_selected = df_wealth2[
    (df_wealth2["Year"] % 5 == 0) &
    (df_wealth2["Date"].str.endswith("Q3"))
].copy()

wealth_shares = (
    df_wealth2_selected
    .groupby(["Date", "Wealth group"], as_index=False)["Net worth"]
    .sum()
)

wealth_shares["stack_order"] = wealth_shares["Wealth group"].map({
    "Top 1%": 0,
    "Next 9%": 1,
    "Middle 40%": 2,
    "Bottom 50%": 3
})

wealth_order = [
    "Bottom 50%",
    "Middle 40%",
    "Next 9%",
    "Top 1%"
]

legend_order = [
    "Top 1%",
    "Next 9%",
    "Middle 40%",
    "Bottom 50%"
]
wealth_shares.head(50)

,Date,Wealth group,Net worth,stack_order
0,1990:Q3,Bottom 50%,3.5,3
1,1990:Q3,Middle 40%,36.6,2
2,1990:Q3,Next 9%,37.4,1
3,1990:Q3,Top 1%,22.4,0
4,1995:Q3,Bottom 50%,3.7,3
5,1995:Q3,Middle 40%,35.1,2
6,1995:Q3,Next 9%,33.9,1
7,1995:Q3,Top 1%,27.3,0
8,2000:Q3,Bottom 50%,3.2,3
9,2000:Q3,Middle 40%,33.6,2


In [ ]:
wealth_shares["Share (%)"] = (
    wealth_shares["Net worth"] /
    wealth_shares.groupby("Date")["Net worth"].transform("sum") * 100
)

In [ ]:
wealth_order = [
    "Bottom 50%",
    "Middle 40%",
    "Next 9%",
    "Top 1%"
]

chart06b = alt.Chart(wealth_shares).mark_bar().encode(
    x=alt.X(
        "Date:N",
        title="Year",
        sort=alt.SortField("Date", order="ascending"),
        axis=alt.Axis(labelAngle=0)
    ),
    y=alt.Y(
        "Share (%):Q",
        title="Share of total household wealth (%)",
        scale=alt.Scale(domain=[0, 100])
    ),
    color=alt.Color(
        "Wealth group:N",
        scale=alt.Scale(
            domain=wealth_order,
            range=["#0E9613", "#9ADF46", "#EE9944", "#A51A0D"]
        ),
        legend=alt.Legend(
        title="Wealth group",
        values=legend_order
        )
    ),
    order=alt.Order("stack_order:Q", sort="descending"),
    tooltip=[
        "Date:N",
        "Wealth group:N",
        alt.Tooltip("Share (%):Q", format=".1f")
    ]
).properties(
    width=750,
    height=400,
    title="Who Owns America's Wealth? Distribution by Wealth Group"
)

chart06b

alt.Chart(...)

In [ ]:
chart06b.save("../img/06b.png", scale_factor=2)

In [ ]:
# Percentage labels inside each stacked bar
labels06b = alt.Chart(wealth_shares).mark_text(
    color="white",
    fontSize=13,
    fontWeight="bold"
).encode(
    x=alt.X(
        "Date:N",
        sort=alt.SortField("Date", order="ascending")
    ),
    y=alt.Y(
        "Share (%):Q",
        stack="center"
    ),
    text=alt.Text(
        "Share (%):Q",
        format=".1f"
    ),
    detail="Wealth group:N",
    order=alt.Order(
        "stack_order:Q",
        sort="descending"
    )
)

# Combine bars and labels
chart06b_labeled = chart06b + labels06b

chart06b_labeled

alt.LayerChart(...)

# Chart 7

In [ ]:
df_071 = pd.read_csv("../data/dataset_2026-10-17IMF.csv")
df_071.head()

/var/folders/3w/plrcmbn551b7m7src59wh39w0000gn/T/ipykernel_41381/415551028.py:1: DtypeWarning: Columns (13: LATEST_ACTUAL_ANNUAL_DATA, 15: BASE_YEAR, 21: LABOR_SECTOR_EMPLOYMENT_TYPE, 24: FISCAL_SECTOR_INSTRUMENTS_INCLUDED_IN_GROSS_AND_NET_DEBT) have mixed types. Specify dtype option on import or set low_memory=False.
  df_071 = pd.read_csv("../data/dataset_2026-10-17IMF.csv")


,DATASET,SERIES_CODE,OBS_MEASURE,COUNTRY,INDICATOR,FREQUENCY,SCALE,UNIT,COUNTRY_UPDATE_DATE,PUBLICATION_DATE,...,2022,2023,2024,2025,2026,2027,2028,2029,2030,2031
0,IMF.RES:WEO(9.0.0),SYR.GGX_NGDP.A,OBS_VALUE,Syrian Arab Republic,"Expenditure, General government, Percent of GDP",Annual,Units,Percent,NaN,2026-10-13T02:00:00Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IMF.RES:WEO(9.0.0),SYR.PPPPC.A,OBS_VALUE,Syrian Arab Republic,"Gross domestic product (GDP), Per capita, purc...",Annual,Units,NaN,NaN,2026-10-13T02:00:00Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IMF.RES:WEO(9.0.0),SYR.GGR_NGDP.A,OBS_VALUE,Syrian Arab Republic,"Revenue, General government, Percent of GDP",Annual,Units,Percent,NaN,2026-10-13T02:00:00Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IMF.RES:WEO(9.0.0),SYR.PCPIE.A,OBS_VALUE,Syrian Arab Republic,"All Items, Consumer price index (CPI), End-of-...",Annual,Units,Index,NaN,2026-10-13T02:00:00Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,IMF.RES:WEO(9.0.0),SYR.PPPGDP.A,OBS_VALUE,Syrian Arab Republic,"Gross domestic product (GDP), Current prices, ...",Annual,Billions,NaN,NaN,2026-10-13T02:00:00Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_072 = pd.read_csv("../data/billionaires.csv")
df_072.head(10)

,name,category,usd_billions,year,as_of,source,source_url
0,Elon Musk,Individual net worth,908.5,2026,10/17/26,Forbes real-time billionaires,https://www.forbes.com/real-time-billionaires/
1,Jeff Bezos,Individual net worth,369.8,2026,10/17/26,Forbes real-time billionaires,https://www.forbes.com/real-time-billionaires/
2,Larry Page,Individual net worth,279.3,2026,10/17/26,Forbes real-time billionaires,https://www.forbes.com/real-time-billionaires/
3,Sergey Brin,Individual net worth,257.1,2026,10/17/26,Forbes real-time billionaires,https://www.forbes.com/real-time-billionaires/
4,Mark Zuckerberg,Individual net worth,243.3,2026,10/17/26,Forbes real-time billionaires,https://www.forbes.com/real-time-billionaires/
5,Michael Dell,Individual net worth,237.0,2026,10/17/26,Forbes real-time billionaires,https://www.forbes.com/real-time-billionaires/
6,Jensen Huang,Individual net worth,185.6,2026,10/17/26,Forbes real-time billionaires,https://www.forbes.com/real-time-billionaires/
7,Larry Ellison,Individual net worth,167.2,2026,10/17/26,Forbes real-time billionaires,https://www.forbes.com/real-time-billionaires/
8,Steve Ballmer,Individual net worth,163.5,2026,10/17/26,Forbes real-time billionaires,https://www.forbes.com/real-time-billionaires/
9,Warren Buffett,Individual net worth,151.1,2026,10/17/26,Forbes real-time billionaires,https://www.forbes.com/real-time-billionaires/


In [ ]:
imf = df_071[
    df_071["SERIES_CODE"].str.contains(
        r"\.(GGX_NGDP|NGDPD)\.A$", regex=True, na=False
    )
][["COUNTRY", "SERIES_CODE", "2026"]].copy()

imf["indicator"] = imf["SERIES_CODE"].str.extract(
    r"\.(GGX_NGDP|NGDPD)\.A$"
)

imf["2026"] = pd.to_numeric(imf["2026"], errors="coerce")

# One row per country
imf_wide = imf.pivot(
    index="COUNTRY",
    columns="indicator",
    values="2026"
).reset_index()

# Projected government expenditure, USD billions
imf_wide["budget_usd_billions"] = (
    imf_wide["NGDPD"] *
    imf_wide["GGX_NGDP"] / 100
)

imf_wide.head()

/var/folders/3w/plrcmbn551b7m7src59wh39w0000gn/T/ipykernel_41381/2126243610.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_071["SERIES_CODE"].str.contains(


indicator,COUNTRY,GGX_NGDP,NGDPD,budget_usd_billions
0,ASEAN-5,19.777,3795.751,750.685675
1,Advanced Economies,41.416,74234.689,30745.038796
2,"Afghanistan, Islamic Republic of",NaN,NaN,NaN
3,Albania,31.741,32.769,10.401208
4,Algeria,35.059,331.894,116.358717


In [ ]:
# added

top10_countries = (
    imf_wide
    .dropna(subset=["budget_usd_billions"])
    .sort_values("budget_usd_billions", ascending=False)
    .head(20)
    .copy()
)

top10_countries = top10_countries[
    ["COUNTRY", "budget_usd_billions"]
]

print("Top 10 Countries by Government Expenditure (2026)")
display(top10_countries.round(1))

Top 10 Countries by Government Expenditure (2026)


indicator,COUNTRY,budget_usd_billions
1,Advanced Economies,30745.0
68,G7,23296.4
55,Emerging Market and Developing Economies,17008.2
199,United States,12482.4
64,European Union (EU),11412.7
63,Euro Area (EA),9706.8
56,Emerging and Developing Asia,9047.1
38,"China, People's Republic of",6974.6
142,Other Advanced Economies (Advanced Economies e...,3966.0
57,Emerging and Developing Europe,2949.1


In [ ]:

from pathlib import Path

# Sort all countries by projected government expenditure
government_spending_2026 = (
    imf_wide[
        ["COUNTRY", "NGDPD", "GGX_NGDP", "budget_usd_billions"]
    ]
    .dropna(subset=["budget_usd_billions"])
    .sort_values("budget_usd_billions", ascending=False)
    .reset_index(drop=True)
)

# Add ranking and USD trillions
government_spending_2026.insert(
    0, "Rank", range(1, len(government_spending_2026) + 1)
)

government_spending_2026["budget_usd_trillions"] = (
    government_spending_2026["budget_usd_billions"] / 1000
)

# Display every country in the notebook
print(government_spending_2026.to_string(index=False))

# Export CSV
output_path = Path("../data/government_spending_2026.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

government_spending_2026.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"\nExported {len(government_spending_2026)} countries to {output_path}")


 Rank                                                                            COUNTRY     NGDPD  GGX_NGDP  budget_usd_billions  budget_usd_trillions
    1                                                                 Advanced Economies 74234.689    41.416         30745.038796             30.745039
    2                                                                                 G7 55470.253    41.998         23296.396855             23.296397
    3                                           Emerging Market and Developing Economies 53769.087    31.632         17008.237600             17.008238
    4                                                                      United States 32642.373    38.240         12482.443435             12.482443
    5                                                                European Union (EU) 22806.152    50.042         11412.654584             11.412655
    6                                                                     Euro Area (EA)

In [ ]:
selected_countries = [
    "United States",
    "China, People's Republic of",
    "Germany",
    "United Kingdom",
    "Japan",
    "Brazil",
    "India",
    "Russian Federation",
    "Australia",
    "Mexico",
    "Saudi Arabia",
    "Indonesia",
    "Colombia",
    "Portugal",
    "South Africa",
    "Egypt, Arab Republic of",
    "Vietnam",
    "Bangladesh"
]

countries = imf_wide[
    imf_wide["COUNTRY"].isin(selected_countries)
].dropna(subset=["budget_usd_billions"]).copy()

countries = countries.rename(columns={
    "COUNTRY": "name",
    "budget_usd_billions": "usd_billions"
})

countries["category"] = "Government expenditure forecast"

chart07_data = pd.concat([
    df_072[["name", "category", "usd_billions"]],
    countries[["name", "category", "usd_billions"]]
], ignore_index=True)

chart07_data = chart07_data.sort_values(
    "usd_billions", ascending=False
)

chart07_data

,name,category,usd_billions
26,United States,Government expenditure forecast,12482.443435
13,"China, People's Republic of",Government expenditure forecast,6974.570651
16,Germany,Government expenditure forecast,2801.578653
25,United Kingdom,Government expenditure forecast,1855.066632
19,Japan,Government expenditure forecast,1625.900078
12,Brazil,Government expenditure forecast,1291.912893
17,India,Government expenditure forecast,1157.037189
22,Russian Federation,Government expenditure forecast,1122.946507
0,Elon Musk,Individual net worth,908.500000
10,Australia,Government expenditure forecast,839.197223


In [ ]:
print(countries[["name", "usd_billions"]])
print("Total observations:", len(chart07_data))

indicator                         name  usd_billions
11                           Australia    839.197223
16                          Bangladesh     53.618143
26                              Brazil   1291.912893
38         China, People's Republic of   6974.570651
39                            Colombia    198.922320
53             Egypt, Arab Republic of    123.631627
72                             Germany   2801.578653
85                               India   1157.037189
86                           Indonesia    249.549843
93                               Japan   1625.900078
122                             Mexico    633.654636
151                           Portugal    167.480437
155                 Russian Federation   1122.946507
159                       Saudi Arabia    379.750481
169                       South Africa    162.047870
198                     United Kingdom   1855.066632
199                      United States  12482.443435
204                            Vietnam    115.

In [ ]:
def make_bubble_chart(data, max_size, title):
    return (
        alt.Chart(data)
        .mark_circle(opacity=0.65, stroke="white", strokeWidth=1)
        .encode(
            x=alt.X(
                "usd_billions:Q",
                title="Net worth / projected annual expenditure (USD billions)",
                scale=alt.Scale(zero=True)
            ),
            y=alt.Y(
                "name:N",
                sort=alt.SortField(
                    field="usd_billions",
                    order="descending"
                ),
                title=None
            ),
            size=alt.Size(
                "usd_billions:Q",
                scale=alt.Scale(range=[100, max_size]),
                legend=None
            ),
            color=alt.Color(
                "category:N",
                scale=alt.Scale(
                    domain=[
                        "Individual net worth",
                        "Government expenditure forecast",
                        "Top 10 combined wealth"
                    ],
                    range=["#A51A0D", "#A0A1A2", "#E6B422"]
                ),
                legend=alt.Legend(title="Category")
            ),
            tooltip=[
                "name:N",
                "category:N",
                alt.Tooltip(
                    "usd_billions:Q",
                    title="USD billions",
                    format=",.1f"
                )
            ]
        )
        .properties(
            width=750,
            height=550,
            title=title
        )
    )

In [ ]:
top10_total = df_072["usd_billions"].sum()

top10_row = pd.DataFrame([{
    "name": "Top 10 Combined",
    "category": "Top 10 combined wealth",
    "usd_billions": top10_total
}])

chart07_data = pd.concat(
    [chart07_data, top10_row],
    ignore_index=True
)

print(f"Top 10 combined: ${top10_total:,.1f} billion")

Top 10 combined: $2,962.4 billion


In [ ]:
chart07a = make_bubble_chart(
    chart07_data,
    max_size=10000,
    title="Billionaire Wealth vs. Government Spending Forecasts (2026)"
)

chart07a

alt.Chart(...)

In [ ]:
bubble_data_v2 = chart07_data[
    chart07_data["name"] != "United States"
].copy()

chart07b = make_bubble_chart(
    bubble_data_v2,
    max_size=16000,
    title="Billionaire Wealth vs. Government Spending Forecasts (2026)"
)

chart07b

alt.Chart(...)

In [ ]:
chart07b.save("../img/07.png", scale_factor=2)

# Chart 8

In [ ]:
df_081 = pd.read_csv(
    "../data/immigrants_2024.csv",
    skiprows=2
)

df_081 = df_081.rename(columns={
    "Year": "year",
    "Immigrants as a Percentage of the U.S. Population": "immigrant_share"
})

df_081["year"] = pd.to_numeric(df_081["year"], errors="coerce")

df_081["immigrant_share"] = pd.to_numeric(
    df_081["immigrant_share"].str.replace("%", "", regex=False),
    errors="coerce"
)

df_081 = df_081.dropna(subset=["year", "immigrant_share"])

df_081.head()

,year,Number of Immigrants,immigrant_share
0,1850.0,"2,244,600",9.7
1,1860.0,"4,138,700",13.2
2,1870.0,"5,567,200",14.4
3,1880.0,"6,679,900",13.3
4,1890.0,"9,249,500",14.8


In [ ]:
df_082 = pd.read_csv("../data/GINIALLRH.csv")
df_082["year"] = pd.to_datetime(df_082["observation_date"]).dt.year
df_082 = df_082.rename(columns={"GINIALLRH": "gini"})

chart08_data = pd.merge(
    df_081[["year", "immigrant_share"]],
    df_082[["year", "gini"]],
    on="year",
    how="inner"
)

chart08_data = (
    chart08_data[chart08_data["year"] >= 1990]
    .sort_values("year")
    .copy()
)

print(chart08_data[["year", "immigrant_share", "gini"]].to_string(index=False))
print("Observations:", len(chart08_data))

  year  immigrant_share  gini
1990.0              7.9 0.428
2000.0             11.1 0.462
2010.0             12.9 0.470
2011.0             13.0 0.477
2012.0             13.0 0.477
2013.0             13.1 0.482
2014.0             13.3 0.480
2015.0             13.5 0.479
2016.0             13.5 0.481
2017.0             13.7 0.489
2018.0             13.7 0.486
2019.0             13.7 0.484
2021.0             13.6 0.494
2022.0             13.9 0.488
2023.0             14.3 0.485
2024.0             14.8 0.489
Observations: 16


In [ ]:
selected_years = [1990, 2000, 2010, 2015, 2019, 2024]

chart08_data = chart08_data[
    chart08_data["year"].isin(selected_years)
].copy()

In [ ]:
base08 = alt.Chart(chart08_data).encode(
    x=alt.X(
        "immigrant_share:Q",
        title="Foreign-born share of U.S. population (%)",
        scale=alt.Scale(zero=False)
    ),
    y=alt.Y(
        "gini:Q",
        title="Household income Gini coefficient",
        scale=alt.Scale(zero=False)
    )
)

line08 = base08.mark_line(
    color="#9CA3AF",
    strokeWidth=2,
    opacity=0.75
).encode(
    order=alt.Order("year:Q", sort="ascending")
)

dots08 = base08.mark_circle(
    size=110,
    stroke="white",
    strokeWidth=1
).encode(
    color=alt.Color(
        "year:Q",
        scale=alt.Scale(scheme="viridis"),
        title="Year"
    ),
    tooltip=[
        alt.Tooltip("year:Q", title="Year", format=".0f"),
        alt.Tooltip("immigrant_share:Q", title="Foreign-born share (%)", format=".1f"),
        alt.Tooltip("gini:Q", title="Gini coefficient", format=".3f")
    ]
)


labels08 = (
    base08
    .mark_text(
        align="left",
        dx=9,
        dy=-9,
        fontSize=10,
        color="#333333"
    )
    .encode(
        text=alt.Text("year:Q", format=".0f")
    )
)


chart08 = (
    (line08 + dots08 + labels08)
    .properties(
        width=700,
        height=450,
        title="Immigration and Household Income Inequality in the U.S."
    )
)

chart08

alt.LayerChart(...)

In [ ]:
base08 = alt.Chart(chart08_data).encode(
    x=alt.X(
        "immigrant_share:Q",
        title="Foreign-born share of U.S. population (%)",
        scale=alt.Scale(zero=False)
    ),
    y=alt.Y(
        "gini:Q",
        title="Household income Gini coefficient",
        scale=alt.Scale(zero=False)
    )
)

line08 = base08.mark_line(
    color="#9CA3AF",
    strokeWidth=2,
    opacity=0.75
).encode(
    order=alt.Order("year:Q", sort="ascending")
)

dots08 = base08.mark_circle(
    size=110,
    stroke="white",
    strokeWidth=1
).encode(
    color=alt.Color(
        "year:Q",
        scale=alt.Scale(scheme="viridis"),
        title="Year"
    ),
    tooltip=[
        alt.Tooltip("year:Q", title="Year", format=".0f"),
        alt.Tooltip("immigrant_share:Q", title="Foreign-born share (%)", format=".1f"),
        alt.Tooltip("gini:Q", title="Gini coefficient", format=".3f")
    ]
)

labels08 = (
    base08.transform_filter(
        alt.FieldOneOfPredicate(
            field="year",
            oneOf=[1990, 2000, 2010, 2015, 2024]
        )
    )
    .mark_text(
        align="left",
        dx=9,
        dy=-9,
        fontSize=12,
        color="#333333"
    )
    .encode(text=alt.Text("year:Q", format=".0f"))
)

chart08 = (
    (line08 + dots08 + labels08)
    .properties(
        width=700,
        height=450,
        title="Immigration and Household Income Inequality in the U.S."
    )
)

chart08

alt.LayerChart(...)

In [ ]:
chart08.save("../img/08.png", scale_factor=2)

#### Immigration and Income Inequality in the United States (1990–2024)
The foreign-born share of the U.S. population and household income inequality both increased substantially between 1990 and 2024, suggesting a positive historical association. However, the relationship varies across periods, particularly after 2021, when the foreign-born share increased while the Gini coefficient declined. Because both variables change over time and may be influenced by other economic and demographic factors, this descriptive relationship should not be interpreted as evidence that immigration causes income inequality.

In [ ]:
print("Immigration years:", sorted(df_081["year"].dropna().unique()))
print("Gini years:", sorted(df_082["year"].dropna().unique()))
print("Merged years:", sorted(chart08_data["year"].dropna().unique()))
print("Observations:", len(chart08_data))

Immigration years: [np.float64(1850.0), np.float64(1860.0), np.float64(1870.0), np.float64(1880.0), np.float64(1890.0), np.float64(1900.0), np.float64(1910.0), np.float64(1920.0), np.float64(1930.0), np.float64(1940.0), np.float64(1950.0), np.float64(1960.0), np.float64(1970.0), np.float64(1980.0), np.float64(1990.0), np.float64(2000.0), np.float64(2010.0), np.float64(2011.0), np.float64(2012.0), np.float64(2013.0), np.float64(2014.0), np.float64(2015.0), np.float64(2016.0), np.float64(2017.0), np.float64(2018.0), np.float64(2019.0), np.float64(2021.0), np.float64(2022.0), np.float64(2023.0), np.float64(2024.0)]
Gini years: [np.int32(1967), np.int32(1968), np.int32(1969), np.int32(1970), np.int32(1971), np.int32(1972), np.int32(1973), np.int32(1974), np.int32(1975), np.int32(1976), np.int32(1977), np.int32(1978), np.int32(1979), np.int32(1980), np.int32(1981), np.int32(1982), np.int32(1983), np.int32(1984), np.int32(1985), np.int32(1986), np.int32(1987), np.int32(1988), np.int32(1989),

In [ ]:
labels08 = (
    base08
    .mark_text(
        align="left",
        dx=9,
        dy=-9,
        fontSize=10,
        color="#333333"
    )
    .encode(
        text=alt.Text("year:Q", format=".0f")
    )
)

In [ ]:
color=alt.Color(
    "year:Q",
    scale=alt.Scale(scheme="viridis"),
    legend=alt.Legend(
        title="Year",
        values=sorted(chart08_data["year"].dropna().unique().tolist())
    )
)